In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import pysam
from Bio.Seq import Seq 
import copy
import warnings
warnings.filterwarnings('ignore')

In [2]:
import io
import itertools

from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components
from alphagenome.visualization.plot_transcripts import TranscriptStyle
from IPython.display import clear_output
from alphagenome.data import track_data

In [ ]:
df = pd.read_excel('./data/3ps.xlsx')

In [4]:
df

,id,WGATAR_id_#,WGATAR_id,WGATAR_chr,WGATAR_start,WGATAR_end,WGATAR_strand,WGATR_motif_seq,type,oligo_230nt_id,oligo_230nt_coordinate,oligo_230nt_chr,oligo_230nt_start,oligo_230nt_end,oligo_230nt_strand,oligo_seq_230nt,Unnamed: 16
0,ps1,2460,WGATAR_2460,chrX,48641413,48641419,-,AGATAA,original,chrX:48641301-48641531(-)_original,chrX:48641301-48641531,chrX,48641301,48641531,-,AGGCTGTGAGGCCAGGCCTCACCCCATTCCCTCCCCCATCCCCGTG...,230
1,ps1,2460,WGATAR_2460,chrX,48641413,48641419,-,AGATAA,modified,chrX:48641301-48641531(-)_modified,chrX:48641301-48641531,chrX,48641301,48641531,-,AGGCTGTGAGGCCAGGCCTCACCCCATTCCCTCCCCCATCCCCGTG...,230
2,ps2,2462,WGATAR_2462,chrX,48644373,48644379,+,TGATAA,original,chrX:48644252-48644482(-)_original,chrX:48644252-48644482,chrX,48644270,48644500,-,TTGATGACCCAGGCTAAGCCTGCAGGCCAGGCCAGTGCTGGCGGGA...,230
3,ps2,2462,WGATAR_2462,chrX,48644373,48644379,+,TGATAA,modified,chrX:48644252-48644482(-)_modified,chrX:48644252-48644482,chrX,48644270,48644500,-,TTGATGACCCAGGCTAAGCCTGCAGGCCAGGCCAGTGCTGGCGGGA...,230
4,ps3,X,X,chrX,48644382,48644388,-,AGATAA,original,chrX:48644261-48644491(+)_original,chrX:48644261-48644491,chrX,48644261,48644491,+,GCAGCATGGCGGGCAAGAAGTTGAGGCCACTGTCCCTGGGTGTTCC...,230
5,ps3,X,X,chrX,48644382,48644388,-,AGATAA,modified,chrX:48644261-48644491(+)_modified,chrX:48644261-48644491,chrX,48644261,48644491,+,GCAGCATGGCGGGCAAGAAGTTGAGGCCACTGTCCCTGGGTGTTCC...,230


In [5]:
# df = pd.read_csv('GATA1_oligo_synthesis_info.csv')
df = df[['WGATAR_id', 'type', 'oligo_230nt_chr',	'oligo_230nt_start', 'oligo_230nt_end',	'oligo_230nt_strand', 'oligo_seq_230nt']]
df = df.rename(columns={'oligo_230nt_chr':'chr', 'oligo_230nt_start':'start', 'oligo_230nt_end':'end', 'oligo_230nt_strand':'strand','oligo_seq_230nt':'seq'})

In [6]:
seq_length = 131072
gata_length = 230
l = (seq_length - gata_length)/2
fasta = pysam.FastaFile('hg19.fa')

In [7]:
seq_length

131072

In [8]:
from pyliftover import LiftOver
lo = LiftOver('hg19', 'hg38')

In [9]:
# GATA_id = ['WGATAR_2460'] 
GATA_id = ['WGATAR_2462'] 

seq_dict = {}
for i in GATA_id: 
    chrom = list(df.loc[(df['WGATAR_id'] == i) & (df['type'] == 'original'), 'chr'])[0]
    s = list(df.loc[(df['WGATAR_id'] == i) & (df['type'] == 'original'), 'start'])[0]
    e = list(df.loc[(df['WGATAR_id'] == i) & (df['type'] == 'original'), 'end'])[0]
    strand = list(df.loc[(df['WGATAR_id'] == i) & (df['type'] == 'original'), 'strand'])[0]

    start_seq = fasta.fetch(chrom, s-l, s)
    start_seq = start_seq.upper()
    end_seq = fasta.fetch(chrom, e, e+l)
    end_seq = end_seq.upper()
    
    so = list(df.loc[(df['WGATAR_id'] == i) & (df['type'] == 'original'), 'seq'])[0]
    so = so.upper()
    so = start_seq + so + end_seq
    sm = list(df.loc[(df['WGATAR_id'] == i) & (df['type'] == 'modified'), 'seq'])[0]
    sm = sm.upper()
    sm2 = start_seq + sm + end_seq

    s_38 = lo.convert_coordinate(chrom, s)[0][1]
    e_38 = lo.convert_coordinate(chrom, e)[0][1]

In [10]:
for i, (a, b) in enumerate(zip(so, sm2)):
    if a != b:
        print(f"Index {i}: '{a}' != '{b}'")

Index 65535: 'A' != 'G'
Index 65537: 'A' != 'G'


In [11]:
so[112]

'G'

In [12]:
sm1_2 = so[:65533] + 'G' + so[65533+1:]
sm1_4 = so[:65535] + 'G' + so[65535+1:]
sm1_6 = so[:65537] + 'G' + so[65537+1:]
sm2 = copy.deepcopy(sm2)
sm3= sm2[:65533] + 'G' + sm2[65533+1:]

In [13]:
for i, (a, b) in enumerate(zip(so, sm3)):
    if a != b:
        print(f"Index {i}: '{a}' != '{b}'")

Index 65533: 'A' != 'G'
Index 65535: 'A' != 'G'
Index 65537: 'A' != 'G'


In [49]:
len(so)

131072

In [ ]:
dna_model = dna_client.create('your_api_key')

In [15]:
BCL11A_interval = genome.Interval(
    chromosome='chrX', start=48776585, end=48796593, strand=strand
)

In [19]:
output_type = {dna_client.OutputType.RNA_SEQ,dna_client.OutputType.CHIP_TF}
output_term = ['EFO:0002067']

In [21]:
output_o = dna_model.predict_sequence(
    sequence=so.center(131072, 'N'),
    requested_outputs=output_type,
    ontology_terms=output_term,
)

output_m1_2 = dna_model.predict_sequence(
    sequence=sm1_2.center(131072, 'N'),
    requested_outputs=output_type,
    ontology_terms=output_term,
)

output_m1_4 = dna_model.predict_sequence(
    sequence=sm1_4.center(131072, 'N'),
    requested_outputs=output_type,
    ontology_terms=output_term,
)

output_m1_6 = dna_model.predict_sequence(
    sequence=sm1_6.center(131072, 'N'),
    requested_outputs=output_type,
    ontology_terms=output_term,
)

output_m2 = dna_model.predict_sequence(
    sequence=sm2.center(131072, 'N'),
    requested_outputs=output_type,
    ontology_terms=output_term,
)

output_m3 = dna_model.predict_sequence(
    sequence=sm3.center(131072, 'N'),
    requested_outputs=output_type,
    ontology_terms=output_term,
)

In [22]:
interval_n = genome.Interval(
    chromosome='chrX', start=int(s_38 - l), end=int(e_38+l), strand=strand
)

In [23]:
# # Load gene annotations (from GENCODE).
# gtf = pd.read_feather(
#     'https://storage.googleapis.com/alphagenome/reference/gencode/'
#     'hg38/gencode.v46.annotation.gtf.gz.feather'
# )

# # Filter to protein-coding genes and highly supported transcripts.
# gtf_transcript = gene_annotation.filter_transcript_support_level(
#     gene_annotation.filter_protein_coding(gtf), ['1']
# )

# # Define an extractor that fetches only the longest transcript per gene.
# gtf_longest_transcript = gene_annotation.filter_to_longest_transcript(
#     gtf_transcript
# )
# longest_transcript_extractor = transcript_utils.TranscriptExtractor(
#     gtf_longest_transcript
# )

In [26]:
rna_ref = output_o.rna_seq.filter_to_unstranded().values
rna_m12 = output_m1_2.rna_seq.filter_to_unstranded().values
rna_m14 = output_m1_4.rna_seq.filter_to_unstranded().values
rna_m16 = output_m1_6.rna_seq.filter_to_unstranded().values
rna_m2 = output_m2.rna_seq.filter_to_unstranded().values
rna_m3 = output_m3.rna_seq.filter_to_unstranded().values

In [27]:
tf_ref = output_o.chip_tf.select_tracks_by_index([87]).values
tf_m12 = output_m1_2.chip_tf.select_tracks_by_index([87]).values
tf_m14 = output_m1_4.chip_tf.select_tracks_by_index([87]).values
tf_m16 = output_m1_6.chip_tf.select_tracks_by_index([87]).values
tf_m2 = output_m2.chip_tf.select_tracks_by_index([87]).values
tf_m3 = output_m3.chip_tf.select_tracks_by_index([87]).values

In [28]:
track_rna_ref = track_data.TrackData(values=rna_ref, metadata=output_o.rna_seq.filter_to_unstranded().metadata, interval=interval_n, 
                                     resolution=output_o.rna_seq.filter_to_unstranded().resolution)
track_rna_m12 = track_data.TrackData(values=rna_m12, metadata=output_o.rna_seq.filter_to_unstranded().metadata, interval=interval_n, 
                                     resolution=output_o.rna_seq.filter_to_unstranded().resolution)
track_rna_m14 = track_data.TrackData(values=rna_m14, metadata=output_o.rna_seq.filter_to_unstranded().metadata, interval=interval_n, 
                                     resolution=output_o.rna_seq.filter_to_unstranded().resolution)
track_rna_m16 = track_data.TrackData(values=rna_m16, metadata=output_o.rna_seq.filter_to_unstranded().metadata, interval=interval_n, 
                                     resolution=output_o.rna_seq.filter_to_unstranded().resolution)
track_rna_m2 = track_data.TrackData(values=rna_m2, metadata=output_o.rna_seq.filter_to_unstranded().metadata, interval=interval_n, 
                                     resolution=output_o.rna_seq.filter_to_unstranded().resolution)
track_rna_m3 = track_data.TrackData(values=rna_m3, metadata=output_o.rna_seq.filter_to_unstranded().metadata, interval=interval_n, 
                                     resolution=output_o.rna_seq.filter_to_unstranded().resolution)


In [29]:
track_tf_ref = track_data.TrackData(values=tf_ref, metadata= output_o.chip_tf.select_tracks_by_index([87]).metadata, interval=interval_n, 
                                     resolution= output_o.chip_tf.select_tracks_by_index([87]).resolution)
track_tf_m12 = track_data.TrackData(values=tf_m12, metadata= output_o.chip_tf.select_tracks_by_index([87]).metadata, interval=interval_n, 
                                     resolution= output_o.chip_tf.select_tracks_by_index([87]).resolution)
track_tf_m14 = track_data.TrackData(values=tf_m14, metadata= output_o.chip_tf.select_tracks_by_index([87]).metadata, interval=interval_n, 
                                     resolution= output_o.chip_tf.select_tracks_by_index([87]).resolution)
track_tf_m16 = track_data.TrackData(values=tf_m16, metadata= output_o.chip_tf.select_tracks_by_index([87]).metadata, interval=interval_n, 
                                     resolution= output_o.chip_tf.select_tracks_by_index([87]).resolution)
track_tf_m2 = track_data.TrackData(values=tf_m2, metadata= output_o.chip_tf.select_tracks_by_index([87]).metadata, interval=interval_n, 
                                     resolution= output_o.chip_tf.select_tracks_by_index([87]).resolution)
track_tf_m3 = track_data.TrackData(values=tf_m3, metadata= output_o.chip_tf.select_tracks_by_index([87]).metadata, interval=interval_n, 
                                     resolution= output_o.chip_tf.select_tracks_by_index([87]).resolution)

In [30]:
rna_ref = track_rna_ref.slice_by_interval(BCL11A_interval)   
rna_m12 = track_rna_m12.slice_by_interval(BCL11A_interval)
rna_m14 = track_rna_m14.slice_by_interval(BCL11A_interval)
rna_m16 = track_rna_m16.slice_by_interval(BCL11A_interval)
rna_m2 = track_rna_m2.slice_by_interval(BCL11A_interval)
rna_m3 = track_rna_m3.slice_by_interval(BCL11A_interval)

In [31]:
tf_ref = track_tf_ref.slice_by_interval(BCL11A_interval, match_resolution=True)   
tf_m12 = track_tf_m12.slice_by_interval(BCL11A_interval, match_resolution=True)
tf_m14 = track_tf_m14.slice_by_interval(BCL11A_interval, match_resolution=True)
tf_m16 = track_tf_m16.slice_by_interval(BCL11A_interval, match_resolution=True)
tf_m2 = track_tf_m2.slice_by_interval(BCL11A_interval, match_resolution=True)
tf_m3 = track_tf_m3.slice_by_interval(BCL11A_interval, match_resolution=True)

In [32]:
rna_ref = rna_ref.values
rna_ref = rna_ref.reshape(-1)

rna_m12 = rna_m12.values
rna_m12 = rna_m12.reshape(-1)

rna_m14 = rna_m14.values
rna_m14 = rna_m14.reshape(-1)

rna_m16 = rna_m16.values
rna_m16 = rna_m16.reshape(-1)

rna_m2 = rna_m2.values
rna_m2 = rna_m2.reshape(-1)

rna_m3 = rna_m3.values
rna_m3 = rna_m3.reshape(-1)

In [33]:
np.save('rna_ref.npy',rna_ref)
np.save('rna_m12.npy',rna_m12)
np.save('rna_m14.npy',rna_m14)
np.save('rna_m16.npy',rna_m16)
np.save('rna_m2.npy',rna_m2)
np.save('rna_m3.npy',rna_m3)

In [35]:
!python array_to_bw.py \
  --npy rna_ref.npy \
  --chrom chrX \
  --start 48634967 \
  --bin-size 1 \
  --chrom-sizes hg19.chrom.sizes \
  --out-bw ./WGATAR_2460/rna_ref.bw \
  --nan-policy skip

!python array_to_bw.py \
  --npy rna_m12.npy \
  --chrom chrX \
  --start 48634967 \
  --bin-size 1 \
  --chrom-sizes hg19.chrom.sizes \
  --out-bw ./WGATAR_2460/rna_m12.bw \
  --nan-policy skip

!python array_to_bw.py \
  --npy rna_m14.npy \
  --chrom chrX \
  --start 48634967 \
  --bin-size 1 \
  --chrom-sizes hg19.chrom.sizes \
  --out-bw ./WGATAR_2460/rna_m14.bw \
  --nan-policy skip

!python array_to_bw.py \
  --npy rna_m16.npy \
  --chrom chrX \
  --start 48634967 \
  --bin-size 1 \
  --chrom-sizes hg19.chrom.sizes \
  --out-bw ./WGATAR_2460/rna_m16.bw \
  --nan-policy skip

!python array_to_bw.py \
  --npy rna_m2.npy \
  --chrom chrX \
  --start 48634967 \
  --bin-size 1 \
  --chrom-sizes hg19.chrom.sizes \
  --out-bw ./WGATAR_2460/rna_m2.bw \
  --nan-policy skip

!python array_to_bw.py \
  --npy rna_m3.npy \
  --chrom chrX \
  --start 48634967 \
  --bin-size 1 \
  --chrom-sizes hg19.chrom.sizes \
  --out-bw ./WGATAR_2460/rna_m3.bw \
  --nan-policy skip

Wrote 20008 bins to ./WGATAR_2460/rna_ref.bw
Wrote 20008 bins to ./WGATAR_2460/rna_m12.bw
Wrote 20008 bins to ./WGATAR_2460/rna_m14.bw
Wrote 20008 bins to ./WGATAR_2460/rna_m16.bw
Wrote 20008 bins to ./WGATAR_2460/rna_m2.bw
Wrote 20008 bins to ./WGATAR_2460/rna_m3.bw


In [36]:
tf_ref = tf_ref.values
tf_ref = tf_ref.reshape(-1)

tf_m12 = tf_m12.values
tf_m12 = tf_m12.reshape(-1)

tf_m14 = tf_m14.values
tf_m14 = tf_m14.reshape(-1)

tf_m16 = tf_m16.values
tf_m16 = tf_m16.reshape(-1)

tf_m2 = tf_m2.values
tf_m2 = tf_m2.reshape(-1)

tf_m3 = tf_m3.values
tf_m3 = tf_m3.reshape(-1)

In [37]:
np.save('tf_ref.npy',tf_ref)
np.save('tf_m12.npy',tf_m12)
np.save('tf_m14.npy',tf_m14)
np.save('tf_m16.npy',tf_m16)
np.save('tf_m2.npy',tf_m2)
np.save('tf_m3.npy',tf_m3)

In [38]:
!python array_to_bw.py \
  --npy tf_ref.npy \
  --chrom chrX \
  --start 48634967 \
  --bin-size 128 \
  --chrom-sizes hg19.chrom.sizes \
  --out-bw ./WGATAR_2460/tf_ref.bw \
  --nan-policy skip

!python array_to_bw.py \
  --npy tf_m12.npy \
  --chrom chrX \
  --start 48634967 \
  --bin-size 128 \
  --chrom-sizes hg19.chrom.sizes \
  --out-bw ./WGATAR_2460/tf_m12.bw \
  --nan-policy skip

!python array_to_bw.py \
  --npy tf_m14.npy \
  --chrom chrX \
  --start 48634967 \
  --bin-size 128 \
  --chrom-sizes hg19.chrom.sizes \
  --out-bw ./WGATAR_2460/tf_m14.bw \
  --nan-policy skip

!python array_to_bw.py \
  --npy tf_m16.npy \
  --chrom chrX \
  --start 48634967 \
  --bin-size 128 \
  --chrom-sizes hg19.chrom.sizes \
  --out-bw ./WGATAR_2460/tf_m16.bw \
  --nan-policy skip

!python array_to_bw.py \
  --npy tf_m2.npy \
  --chrom chrX \
  --start 48634967 \
  --bin-size 128 \
  --chrom-sizes hg19.chrom.sizes \
  --out-bw ./WGATAR_2460/tf_m2.bw \
  --nan-policy skip

!python array_to_bw.py \
  --npy tf_m3.npy \
  --chrom chrX \
  --start 48634967 \
  --bin-size 128 \
  --chrom-sizes hg19.chrom.sizes \
  --out-bw ./WGATAR_2460/tf_m3.bw \
  --nan-policy skip

Wrote 157 bins to ./WGATAR_2460/tf_ref.bw
Wrote 157 bins to ./WGATAR_2460/tf_m12.bw
Wrote 157 bins to ./WGATAR_2460/tf_m14.bw
Wrote 157 bins to ./WGATAR_2460/tf_m16.bw
Wrote 157 bins to ./WGATAR_2460/tf_m2.bw
Wrote 157 bins to ./WGATAR_2460/tf_m3.bw
